# 01 — BCRD Exchange Rates | Silver Transformation

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Transform the source-preserving Bronze USD/DOP data into a typed, validated and deduplicated Silver table that supports KPIs **FX-001 to FX-009** without further reprocessing.

## Lineage
| Layer | Object |
|---|---|
| Source | `rd_economic_intelligence.bronze.bcrd_exchange_rates_daily` |
| Target | `rd_economic_intelligence.silver.bcrd_exchange_rates_daily` |
| Rejected records | `rd_economic_intelligence.quality.dq_rejected_records` |
| Rule results | `rd_economic_intelligence.quality.dq_rule_results` |

## Grain
One row per business day (`rate_date`) for the `USD/DOP` currency pair. Rates are DOP per 1 USD.

## Transformations
1. Build `rate_date` from the source year, month and day. Bronze holds text in two layouts: the month is a number (`historico_5y`) or a Spanish abbreviation `Ene`…`Dic` (`daily_since_1991`).
2. Cast rates to `DECIMAL(10,4)` with business names (`buy_rate`, `sell_rate`).
3. Derive `spread` (sell − buy) and `mid_rate` ((buy + sell) / 2).
4. Keep BCRD official appreciation/depreciation values as **reference** columns (`bcrd_*_ref_pct`), taken from the newest snapshot that publishes them (only the `historico_5y` layout does; the daily file since 1991 has no such columns). They use the reciprocal method and the opposite sign of a conventional % change, so they are never used as the primary KPI (**decision D1**).
5. Merge all Bronze snapshots (one per archived workbook; each BCRD file covers only five years). For each `rate_date` the **newest file** wins; the same date in an older file is counted as `superseded_by_newer_file` (observational), not rejected. A date repeated **inside one file** is rejected (`unique_rate_date`).
6. Enforce data-quality rules. Failing rows go to `quality.dq_rejected_records`; they are not silently dropped.

## Data-quality rules (enforced)
| Rule | Condition |
|---|---|
| `valid_rate_date` | `rate_date` is a valid calendar date |
| `positive_buy_rate` | `buy_rate > 0` |
| `positive_sell_rate` | `sell_rate > 0` |
| `buy_not_above_sell` | `buy_rate <= sell_rate` |
| `unique_rate_date` | one row per `rate_date` after deduplication |

## Load pattern
Bronze is incremental (Auto Loader); Silver is a full refresh from all Bronze snapshots (overwrite). The data is small (about 1,300 rows per file), so a full rebuild is simpler and fully idempotent. Reconciliation: Bronze = Silver + rejected + superseded.

**Status:** Development — not production-ready.

In [0]:
# ============================================================
# Step 1 — Configuration
# ============================================================
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE_TABLE = f"{CATALOG}.bronze.bcrd_exchange_rates_daily"
TARGET_TABLE = f"{CATALOG}.silver.bcrd_exchange_rates_daily"
REJECTED_TABLE = f"{CATALOG}.quality.dq_rejected_records"
RULE_RESULTS_TABLE = f"{CATALOG}.quality.dq_rule_results"

CURRENCY_PAIR = "USD/DOP"
RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)

print(f"Run ID: {RUN_ID}")
print(f"Source: {SOURCE_TABLE}")
print(f"Target: {TARGET_TABLE}")

In [0]:
# ============================================================
# Step 2 — Create the quality tables (first run only)
# ============================================================
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {REJECTED_TABLE} (
        run_id           STRING    COMMENT 'Pipeline run that detected the failure',
        target_table     STRING    COMMENT 'Table the record was meant for',
        rule_name        STRING    COMMENT 'Data-quality rule that failed',
        record_key       STRING    COMMENT 'Business key of the record, when available',
        record_json      STRING    COMMENT 'Full record as JSON, for investigation',
        source_batch_id  STRING    COMMENT 'Bronze batch the record came from',
        detected_ts      TIMESTAMP COMMENT 'When the failure was detected'
    )
    COMMENT 'Records rejected by data-quality rules. Rejected is not deleted: rows stay here for review.'
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {RULE_RESULTS_TABLE} (
        run_id         STRING    COMMENT 'Pipeline run',
        target_table   STRING    COMMENT 'Table the rule protects',
        rule_name      STRING    COMMENT 'Data-quality rule',
        rule_type      STRING    COMMENT 'enforced = failing rows rejected; observational = reported only',
        records_tested BIGINT    COMMENT 'Rows evaluated',
        records_failed BIGINT    COMMENT 'Rows that failed',
        status         STRING    COMMENT 'PASS, FAIL or WARN',
        run_ts         TIMESTAMP COMMENT 'Run timestamp'
    )
    COMMENT 'Data-quality rule results per pipeline run.'
""")

print("Quality tables ready")

In [0]:
# ============================================================
# Step 3 — Read Bronze and apply typing / business naming
# ============================================================
bronze = spark.table(SOURCE_TABLE)
bronze_count = bronze.count()

# Month: number (historico_5y) or Spanish abbreviation (daily_since_1991)
MONTH_ABBR = {"ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6, "jul": 7, "ago": 8, "sep": 9, "set": 9, "oct": 10, "nov": 11, "dic": 12}
abbr = F.create_map(*[F.lit(x) for kv in MONTH_ABBR.items() for x in kv])
month = F.when(F.trim("mes").rlike(r"^\d{1,2}$"), F.trim("mes").cast("int")).otherwise(abbr[F.lower(F.substring(F.trim("mes"), 1, 3))])
iso = F.concat_ws("-", F.trim("anio"), F.lpad(month.cast("string"), 2, "0"), F.lpad(F.trim("dia"), 2, "0"))
try_dec = lambda c, t: F.expr(f"try_cast({c} AS {t})")

typed = bronze.select(
    F.to_date(F.call_function("try_to_timestamp", iso, F.lit("yyyy-MM-dd"))).alias("rate_date"),
    F.lit(CURRENCY_PAIR).alias("currency_pair"),
    try_dec("compra", "decimal(10,4)").alias("buy_rate"),
    try_dec("venta", "decimal(10,4)").alias("sell_rate"),
    # BCRD official reference values (reciprocal method, D1) — not primary KPIs
    try_dec("compra_apre_depre_acumulada_pct", "decimal(8,2)").alias("bcrd_buy_ytd_apre_depre_ref_pct"),
    try_dec("compra_apre_depre_interanual_pct", "decimal(8,2)").alias("bcrd_buy_yoy_apre_depre_ref_pct"),
    try_dec("venta_apre_depre_acumulada_pct", "decimal(8,2)").alias("bcrd_sell_ytd_apre_depre_ref_pct"),
    try_dec("venta_apre_depre_interanual_pct", "decimal(8,2)").alias("bcrd_sell_yoy_apre_depre_ref_pct"),
    # Lineage
    F.col("_batch_id").alias("_bronze_batch_id"),
    F.col("_source_file"),
    F.col("_source_row_number"),
    F.col("_ingestion_ts").alias("_bronze_ingestion_ts"),
    F.col("_source_path"),
    F.col("_source_modified_ts"),
)

print(f"Bronze rows read: {bronze_count:,}")
display(typed.limit(5))

In [0]:
# ============================================================
# Step 4 — Evaluate enforced data-quality rules
# ============================================================
rules = {
    "valid_rate_date":    F.col("rate_date").isNotNull(),
    "positive_buy_rate":  F.col("buy_rate").isNotNull() & (F.col("buy_rate") > 0),
    "positive_sell_rate": F.col("sell_rate").isNotNull() & (F.col("sell_rate") > 0),
    "buy_not_above_sell": F.col("buy_rate") <= F.col("sell_rate"),
}

# Tag each row with the first rule it fails (NULL = passes all row-level rules)
failed_rule = F.lit(None).cast("string")
for name, condition in reversed(list(rules.items())):
    failed_rule = F.when(~F.coalesce(condition, F.lit(False)), F.lit(name)).otherwise(failed_rule)

evaluated = typed.withColumn("_failed_rule", failed_rule)
valid = evaluated.filter(F.col("_failed_rule").isNull()).drop("_failed_rule")
row_rejects = evaluated.filter(F.col("_failed_rule").isNotNull())

In [0]:
# ============================================================
# Step 5 — Deduplicate by business key (rate_date)
# ============================================================
# Inside one file: a repeated date is a source defect -> rejected, not lost.
w_file = Window.partitionBy("rate_date", "_source_path").orderBy(F.col("_source_row_number"))
in_file = valid.withColumn("_rank", F.row_number().over(w_file))
duplicate_rejects = in_file.filter("_rank > 1").drop("_rank").withColumn("_failed_rule", F.lit("unique_rate_date"))
unique_per_file = in_file.filter("_rank = 1").drop("_rank")

# Across files: the newest archived file wins; older copies are superseded (expected overlap of rolling 5-year files).
w_new = Window.partitionBy("rate_date").orderBy(F.col("_source_modified_ts").desc(), F.col("_source_path").desc())
ranked = unique_per_file.withColumn("_rank", F.row_number().over(w_new))
REF_COLS = ["bcrd_buy_ytd_apre_depre_ref_pct", "bcrd_buy_yoy_apre_depre_ref_pct", "bcrd_sell_ytd_apre_depre_ref_pct", "bcrd_sell_yoy_apre_depre_ref_pct"]
winners = ranked.filter("_rank = 1").drop("_rank", "_source_path", "_source_modified_ts", *REF_COLS)
# Reference columns: from the newest snapshot that publishes them for that date (only the historico_5y layout has them)
refs = (unique_per_file.filter(F.col(REF_COLS[0]).isNotNull())
        .withColumn("_r", F.row_number().over(w_new)).filter("_r = 1").select("rate_date", *REF_COLS))
deduplicated = winners.join(refs, "rate_date", "left")
superseded_count = ranked.filter("_rank > 1").count()
# Observational: the same date in two files should carry the same rates
dup_dates = ranked.groupBy("rate_date").agg(F.countDistinct("buy_rate").alias("nb"), F.countDistinct("sell_rate").alias("ns"), F.count("*").alias("n")).filter("n > 1")
overlap_dates = dup_dates.count()
rate_conflicts = dup_dates.filter("nb > 1 OR ns > 1").count()
snapshots = valid.select("_source_path").distinct().count()
print(f"Snapshots merged: {snapshots} · superseded by a newer file: {superseded_count:,}")

In [0]:
# ============================================================
# Step 6 — Derive analytical columns
# ============================================================
silver = (
    deduplicated
    .withColumn("spread", (F.col("sell_rate") - F.col("buy_rate")).cast("decimal(10,4)"))
    .withColumn("mid_rate", ((F.col("buy_rate") + F.col("sell_rate")) / 2).cast("decimal(12,6)"))
    .withColumn("year", F.year("rate_date"))
    .withColumn("month", F.month("rate_date"))
    .withColumn("_silver_run_id", F.lit(RUN_ID))
    .withColumn("_silver_processed_ts", F.lit(RUN_TS).cast("timestamp"))
    .select(
        "rate_date", "currency_pair", "buy_rate", "sell_rate", "spread", "mid_rate",
        "year", "month",
        "bcrd_buy_ytd_apre_depre_ref_pct", "bcrd_buy_yoy_apre_depre_ref_pct",
        "bcrd_sell_ytd_apre_depre_ref_pct", "bcrd_sell_yoy_apre_depre_ref_pct",
        "_bronze_batch_id", "_source_file", "_source_row_number", "_bronze_ingestion_ts",
        "_silver_run_id", "_silver_processed_ts",
    )
    .orderBy("rate_date")
)

In [0]:
# ============================================================
# Step 7 — Write Silver (full refresh) and record rejected rows
# ============================================================
(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TARGET_TABLE)
)

all_rejects = row_rejects.drop("_source_path", "_source_modified_ts").unionByName(duplicate_rejects.drop("_source_path", "_source_modified_ts"))
rejected_count = all_rejects.count()

if rejected_count > 0:
    (
        all_rejects.select(
            F.lit(RUN_ID).alias("run_id"),
            F.lit(TARGET_TABLE).alias("target_table"),
            F.col("_failed_rule").alias("rule_name"),
            F.col("rate_date").cast("string").alias("record_key"),
            F.to_json(F.struct(*[c for c in typed.columns if c not in ("_source_path", "_source_modified_ts")])).alias("record_json"),
            F.col("_bronze_batch_id").alias("source_batch_id"),
            F.lit(RUN_TS).cast("timestamp").alias("detected_ts"),
        )
        .write.mode("append").saveAsTable(REJECTED_TABLE)
    )

silver_count = spark.table(TARGET_TABLE).count()
print(f"Silver rows written:  {silver_count:,}")
print(f"Rejected rows logged: {rejected_count:,}")

In [0]:
# ============================================================
# Step 8 — Table documentation and Delta constraints
# ============================================================
spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Silver: BCRD official USD/DOP daily buy/sell rates. One row per business day, typed, validated and deduplicated. Supports KPIs FX-001 to FX-009.'
""")

column_comments = {
    "rate_date": "Business day of the published rate",
    "currency_pair": "Currency pair; rates are DOP per 1 USD",
    "buy_rate": "BCRD reference buy rate (FX-001)",
    "sell_rate": "BCRD reference sell rate (FX-002)",
    "spread": "sell_rate - buy_rate",
    "mid_rate": "(buy_rate + sell_rate) / 2",
    "bcrd_buy_ytd_apre_depre_ref_pct": "BCRD official accumulated appreciation(+)/depreciation(-) of buy rate, reciprocal method. Reference only (D1)",
    "bcrd_buy_yoy_apre_depre_ref_pct": "BCRD official year-on-year appreciation(+)/depreciation(-) of buy rate, reciprocal method. Reference only (D1)",
    "bcrd_sell_ytd_apre_depre_ref_pct": "BCRD official accumulated appreciation(+)/depreciation(-) of sell rate, reciprocal method. Reference only (D1)",
    "bcrd_sell_yoy_apre_depre_ref_pct": "BCRD official year-on-year appreciation(+)/depreciation(-) of sell rate, reciprocal method. Reference only (D1)",
}
for column, comment in column_comments.items():
    safe = comment.replace("'", "\\'")
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN {column} COMMENT '{safe}'")

# Delta CHECK constraints: future writes that break the rules will fail at the table level
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN rate_date SET NOT NULL")
for name, expr in {
    "chk_positive_buy_rate": "buy_rate > 0",
    "chk_positive_sell_rate": "sell_rate > 0",
    "chk_buy_not_above_sell": "buy_rate <= sell_rate",
}.items():
    spark.sql(f"ALTER TABLE {TARGET_TABLE} DROP CONSTRAINT IF EXISTS {name}")
    spark.sql(f"ALTER TABLE {TARGET_TABLE} ADD CONSTRAINT {name} CHECK ({expr})")

print("Comments and constraints applied")

In [0]:
# ============================================================
# Step 9 — Record rule results and verify
# ============================================================
results = [
    (name, "enforced", bronze_count, row_rejects.filter(F.col("_failed_rule") == name).count())
    for name in rules
]
results.append(("unique_rate_date", "enforced", valid.count(), duplicate_rejects.count()))
results.append(("superseded_by_newer_file", "observational", unique_per_file.count(), superseded_count))
results.append(("rates_match_across_snapshots", "observational", overlap_dates, rate_conflicts))

# Observational checks (reported, not enforced)
weekend_rows = spark.table(TARGET_TABLE).filter(F.dayofweek("rate_date").isin(1, 7)).count()
gaps = (
    spark.table(TARGET_TABLE)
    .withColumn("_prev", F.lag("rate_date").over(Window.orderBy("rate_date")))
    .filter(F.datediff("rate_date", "_prev") > 4)
    .count()
)
results.append(("no_weekend_dates", "observational", silver_count, weekend_rows))
results.append(("no_gap_over_4_calendar_days", "observational", silver_count, gaps))

rows = [
    (RUN_ID, TARGET_TABLE, name, rtype, int(tested), int(failed),
     "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN"), RUN_TS)
    for name, rtype, tested, failed in results
]
spark.createDataFrame(
    rows,
    "run_id string, target_table string, rule_name string, rule_type string, "
    "records_tested bigint, records_failed bigint, status string, run_ts timestamp",
).write.mode("append").saveAsTable(RULE_RESULTS_TABLE)

print("Data-quality results:")
for r in rows:
    print(f"  {r[6]:<4} {r[2]:<28} ({r[3]}) tested={r[4]:,} failed={r[5]:,}")

# Reconciliation: every Bronze row is in Silver, in the rejected log, or superseded by a newer file
assert silver_count + rejected_count + superseded_count == bronze_count, (
    f"Reconciliation failed: silver {silver_count} + rejected {rejected_count} + superseded {superseded_count} != bronze {bronze_count}"
)
print(f"PASS: reconciliation — {silver_count:,} silver + {rejected_count:,} rejected + {superseded_count:,} superseded = {bronze_count:,} bronze")

summary = spark.sql(f"""
    SELECT COUNT(*) AS rows, MIN(rate_date) AS first_date, MAX(rate_date) AS last_date,
           MIN(buy_rate) AS min_buy, MAX(buy_rate) AS max_buy,
           ROUND(AVG(spread), 4) AS avg_spread
    FROM {TARGET_TABLE}
""")
display(summary)

In [0]:
# Return a run summary (visible in Jobs run output when executed as a task)
import json
s = summary.first()
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "target_table": TARGET_TABLE,
    "bronze_rows": bronze_count,
    "silver_rows": silver_count,
    "rejected_rows": rejected_count,
    "snapshots": snapshots,
    "superseded_by_newer_file": superseded_count,
    "overlap_dates": overlap_dates,
    "rate_conflicts_across_snapshots": rate_conflicts,
    "first_date": str(s["first_date"]),
    "last_date": str(s["last_date"]),
    "min_buy": str(s["min_buy"]),
    "max_buy": str(s["max_buy"]),
    "avg_spread": str(s["avg_spread"]),
    "dq": {r[2]: f"{r[6]} ({r[5]} failed)" for r in rows},
}))